# MemMachine Memory Integration with NeMo Agent toolkit

This notebook demonstrates how to use MemMachine memory inside NeMo Agent toolkit end-to-end. MemMachine provides a unified memory management system where users can add conversations or memories directly.

## What You'll Learn

- How to connect to a running MemMachine Docker service
- How to integrate MemMachine memory with NeMo Agent toolkit
- How to add and retrieve memories from conversations
- How to add and retrieve memories directly
- How to use memory in an agent workflow with tools

<a id="installing-deps"></a>
### Step 0.1) Installing Dependencies

**Note:** This notebook should be run from the **root of the repository** for the installation to work correctly.

The cell below installs `nvidia-nat[langchain]` (from the root `pyproject.toml`) and `nvidia-nat-memmachine` (from `packages/nvidia_nat_memmachine`). Running from the repository root is required so both paths resolve correctly. MemMachine itself runs as a Docker service — see `examples/deploy/README.md` for setup instructions.

In [ ]:
%%bash
uv pip show -q "nvidia-nat-langchain"
nat_langchain_installed=$?
uv pip show -q "nvidia-nat-memmachine"
nat_memmachine_installed=$?

if [[ ${nat_langchain_installed} -ne 0 ]]; then
    echo "Installing nvidia-nat with LangChain support..."
    uv pip install -e ".[langchain]"
else
    echo "✓ nvidia-nat[langchain] is already installed"
fi

if [[ ${nat_memmachine_installed} -ne 0 ]]; then
    echo "Installing nvidia-nat-memmachine..."
    uv pip install -e ".[memmachine]"
else
    echo "✓ nvidia-nat-memmachine is already installed"
fi

In [ ]:
# Ensure nat command uses the kernel's Python environment
import os
import sys

venv_bin = os.path.dirname(sys.executable)
if venv_bin not in os.environ['PATH'].split(os.pathsep)[0]:
    os.environ['PATH'] = f"{venv_bin}{os.pathsep}{os.environ['PATH']}"
    print(f"✓ Added {venv_bin} to PATH")
else:
    print(f"✓ {venv_bin} already in PATH")

### Step 0.2) Configure API Keys

You'll need an **NVIDIA API key** to use NVIDIA models in this notebook.

- Get your key from [NVIDIA Build](https://build.nvidia.com/explore/discover)
- Navigate to any model page and click "Get API Key"

**Note:** MemMachine's LLM provider keys (e.g. `OPENAI_API_KEY`) are configured via environment variables or `configuration.yml` before starting the Docker stack — not here.

In [ ]:
import os
from getpass import getpass

# Prompt for NVIDIA API key
if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key
    print("✓ NVIDIA API key set")
else:
    print("✓ NVIDIA API key already set in environment")

#### Step 0.3: Configure Server Port

Run the cell below to enter the host port that MemMachine is exposed on in your Docker setup.

- The default port in `docker-compose.memmachine.yml` is **8095** — just press **Enter** to use it
- If you mapped a different host port (e.g. `MEMORY_SERVER_PORT=9000`), enter that number

**Note:** Make sure the MemMachine Docker stack is already running before continuing. See `examples/deploy/README.md` for start/stop instructions.

In [ ]:
# Prompt for the MemMachine server port
prompt = ("Enter the host port MemMachine is running on in Docker "
          "(default is 8095, press Enter to use default): ")
port_input = input(prompt).strip()

if port_input:
    MEMMACHINE_PORT = int(port_input)
else:
    MEMMACHINE_PORT = 8095

MEMMACHINE_BASE_URL = f"http://localhost:{MEMMACHINE_PORT}"
print(f"✓ MemMachine server will use: {MEMMACHINE_BASE_URL}")

<a id="basic-memory"></a>
## 1) Basic Memory Operations

Let's explore how to use MemMachine memory programmatically with NeMo Agent toolkit.

<a id="programmatic"></a>
### 1.1) Programmatic Memory Usage

First, let's import the necessary modules:

In [ ]:
import asyncio
import uuid

from nat.builder.workflow_builder import WorkflowBuilder
from nat.data_models.config import GeneralConfig
from nat.memory.models import MemoryItem
from nat.plugins.memmachine.memory import MemMachineMemoryClientConfig

# Create a unique test ID for this session
test_id = str(uuid.uuid4())[:8]
print(f"Test session ID: {test_id}")

Now, let's configure and create a MemMachine memory client:

In [ ]:
# Configure MemMachine memory client
memmachine_config = MemMachineMemoryClientConfig(
    base_url=MEMMACHINE_BASE_URL,
    org_id=f"demo_org_{test_id}",
    project_id=f"demo_project_{test_id}",
    timeout=30,
    max_retries=3
)

print("✅ MemMachine configuration created")
print(f"   Base URL: {memmachine_config.base_url}")
print(f"   Org ID: {memmachine_config.org_id}")
print(f"   Project ID: {memmachine_config.project_id}")

<a id="conversation-memory"></a>
### 1.2) Adding Memories from Conversations

Memories can be added from conversations, preserving the full context of interactions. All memories are added to both episodic and semantic memory types. Let's add a memory from a conversation:

In [ ]:
async def add_memory_from_conversation():
    """Add a memory from a conversation"""
    general_config = GeneralConfig()

    async with WorkflowBuilder(general_config=general_config) as builder:
        # Add MemMachine memory client
        await builder.add_memory_client("memmachine_memory", memmachine_config)
        memory_client = await builder.get_memory_client("memmachine_memory")

        # Create a memory with conversation context
        user_id = f"demo_user_{test_id}"
        conversation = [
            {"role": "user", "content": "I love pizza and Italian food, especially margherita pizza."},
            {
                "role": "assistant",
                "content": "I'll remember that you love pizza and Italian food, "
                "especially margherita pizza.",
            },
        ]

        memory_item = MemoryItem(
            conversation=conversation,
            user_id=user_id,
            memory="User loves pizza and Italian food, especially margherita pizza",
            metadata={
                "session_id": f"session_{test_id}",
                "agent_id": f"agent_{test_id}",
                "test_id": "conversation_demo"
            },
            tags=["food", "preference", "italian"]
        )

        # Add the memory
        await memory_client.add_items([memory_item])
        print(f"✅ Added memory from conversation for user: {user_id}")

        # Wait a moment for indexing
        await asyncio.sleep(2)

        return user_id, memory_client

# Run the async function
user_id, memory_client = await add_memory_from_conversation()

<a id="direct-memory"></a>
### 1.3) Adding Memories Directly

Memories can also be added directly without a conversation. All memories are added to both episodic and semantic memory types. These are great for storing long-term user preferences and facts:

In [ ]:
async def add_memory_directly():
    """Add a memory directly (without conversation) using the existing memory client"""
    # Reuse the memory_client from the previous cell
    # This avoids trying to create the project again

    # Create a memory directly (without conversation)
    direct_memory = MemoryItem(
        conversation=None,  # No conversation for direct memory
        user_id=user_id,
        memory="User prefers working in the morning (9 AM - 12 PM) and is allergic to peanuts",
        metadata={
            "session_id": f"session_{test_id}",
            "agent_id": f"agent_{test_id}",
            "test_id": "direct_demo"
        },
        tags=["preference", "allergy", "schedule"]
    )

    # Add the memory using the existing memory_client
    await memory_client.add_items([direct_memory])
    print(f"✅ Added memory directly for user: {user_id}")

    # Direct memories are processed asynchronously
    # Wait longer for background ingestion task
    print("⏳ Waiting for memory ingestion (this may take 2-5 seconds)...")
    await asyncio.sleep(5)

    return memory_client

memory_client = await add_memory_directly()

<a id="searching"></a>
### 1.4) Searching Memories

Now let's search for the memories we just added. **Note:** MemMachine's search function returns all memories in a single search call, whether they were added from conversations or directly.

In [ ]:
async def search_memories():
    """Search for memories - returns all memories in one call"""
    # Reuse the memory_client from the previous cell
    # This avoids trying to create the project again

    # Single search returns all memories
    print("🔍 Searching for memories (pizza/Italian food)...")
    print("   Note: This search returns all memories (from conversations and direct)\n")

    all_results = await memory_client.search(
        query="pizza Italian food margherita",
        top_k=10,
        user_id=user_id,
        session_id=f"session_{test_id}",
        agent_id=f"agent_{test_id}"
    )

    print(f"   Found {len(all_results)} total memories\n")

    # Display results - memories from conversations have conversation field, direct memories don't
    for i, mem in enumerate(all_results, 1):
        memory_type = "From Conversation" if mem.conversation else "Direct"
        print(f"   {i}. [{memory_type}] {mem.memory}")
        if mem.conversation:
            print(f"      Conversation: {len(mem.conversation)} messages")
        if mem.tags:
            print(f"      Tags: {', '.join(mem.tags)}")
        print()

    # Now search for direct memory (may need retries due to async processing)
    print("\n🔍 Searching for direct memory (morning work allergy)...")
    print("   Note: Direct memories may take a few seconds to be searchable\n")

    direct_results = []
    for attempt in range(3):
        direct_results = await memory_client.search(
            query="morning work schedule allergy peanuts",
            top_k=10,
            user_id=user_id,
            session_id=f"session_{test_id}",
            agent_id=f"agent_{test_id}"
        )
        # Filter for direct memories (no conversation)
        direct_only = [m for m in direct_results if not m.conversation]
        if len(direct_only) > 0:
            direct_results = direct_only
            break
        print(f"   Attempt {attempt + 1}: No direct memory results yet, waiting...")
        await asyncio.sleep(2)

    print(f"   Found {len(direct_results)} direct memories")
    for i, mem in enumerate(direct_results, 1):
        print(f"   {i}. {mem.memory}")
        if mem.tags:
            print(f"      Tags: {', '.join(mem.tags)}")

await search_memories()

<a id="agent-workflow"></a>
## 2) Agent Workflow with Memory

Now let's create an agent workflow that can use memory tools to remember and recall information.

<a id="config"></a>
### 2.1) Create Configuration

Let's create a YAML configuration file for an agent with memory capabilities:

**Note:** The `react_agent` workflow type requires `nvidia-nat[langchain]` to be installed. This is because the ReAct agent uses LangChain/LangGraph for its agent framework. If you encounter errors about missing `langchain` modules (like `No module named 'langchain.schema'`), make sure you've run the installation cell above that installs `nvidia-nat[langchain]`. The ReAct agent is one of several agent types available in NeMo Agent toolkit - it requires LangChain because it uses LangGraph for agent orchestration.

In [ ]:
# Write the agent config file using the same test_id from Part 1
# This ensures the agent uses the same MemMachine project we created earlier

agent_config = f'''general:
  telemetry:
    enabled: false

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.7
    max_tokens: 1024

memory:
  memmachine_memory:
    _type: memmachine_memory
    base_url: "{MEMMACHINE_BASE_URL}"
    org_id: "{memmachine_config.org_id}"
    project_id: "{memmachine_config.project_id}"

functions:
  get_memory:
    _type: get_memory
    memory: memmachine_memory
    description: |
      Retrieve memories relevant to a query. Always call this tool first to check
      for existing user preferences or facts.
      Use the exact JSON format with user_id, query, and top_k parameters.

  add_memory:
    _type: add_memory
    memory: memmachine_memory
    description: |
      Add facts about user preferences or information to long-term memory.
      Use the exact JSON format with user_id, memory, conversation (optional), metadata, and tags.

workflow:
  _type: react_agent
  tool_names: [add_memory, get_memory]
  description: "A chat agent that can remember and recall user preferences using MemMachine memory"
  llm_name: nim_llm
  verbose: true
  max_tool_calls: 5
  system_prompt: |
    You are a helpful assistant with access to memory tools. Always use user_id "{user_id}" for memory operations.

    {{tools}}

    Use this format:

    Question: the input question you must answer
    Thought: think about what to do
    Action: the action to take, one of [{{tool_names}}]
    Action Input: {{{{"key": "value"}}}}
    Observation: the result of the action
    ... (repeat Thought/Action/Action Input/Observation as needed)
    Thought: I now know the final answer
    Final Answer: your final answer

    CRITICAL: Action Input must be ONLY valid JSON on a single line. No extra text before or after the JSON.
'''

# Write the config file
with open('memmachine_agent_config.yml', 'w') as f:
    f.write(agent_config)

print(f"✅ Created agent config with project: {memmachine_config.org_id}/{memmachine_config.project_id}")
print("   Config file: memmachine_agent_config.yml")

### 2.2) Run Agent

Now we can run the agent with the MemMachine memory backend. The `!nat run` command uses the correct environment since we configured PATH earlier.

In [ ]:
!nat run --config_file memmachine_agent_config.yml --input "What is my favorite food?"

Great! The agent should have retrieved the food preference memory we added earlier. Now let's tell the agent about another preference so it can add it to memory:

In [ ]:
!nat run --config_file memmachine_agent_config.yml \
    --input "I love reading science fiction novels like Dune, can you recommend some other books in the genre?"

Now let's verify the agent can recall our book preference from memory:

In [ ]:
!nat run --config_file memmachine_agent_config.yml \
    --input "What other books do you think I would like? Also recommend some movies in the genre."

<a id="next-steps"></a>
## 3) Next Steps

Congratulations! You've successfully integrated MemMachine memory with NeMo Agent toolkit. Here are some next steps to explore:

1. **Explore Advanced Memory Features**:
   - Use metadata and tags for better memory organization
   - Experiment with different ways to add memories (from conversations vs directly)
   - Try memory deletion and cleanup strategies

2. **Integrate with Other Components**:
   - Combine memory with RAG (Retrieval Augmented Generation)
   - Use memory in multi-agent workflows
   - Add memory to custom tools and functions

3. **Production Considerations**:
   - Set up proper Neo4j database management
   - Configure memory retention policies
   - Implement memory search optimization
   - Add monitoring and observability

4. **Additional Resources**:
   - [MemMachine Documentation](https://docs.memmachine.ai/)
   - [NeMo Agent toolkit Documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/)
